# RAG-Only Baseline And Failure Modes

This notebook builds a local retriever and then uses a provider call for grounded answer generation. RAG adds evidence to the prompt; it does not train model weights.


## Learning Objectives

By the end of this notebook, you should be able to:

- Build a cited RAG answer path on a sparse retriever.
- Abstain when retrieved evidence is too weak.
- Validate citation IDs against the retrieved context.
- Measure retrieval with recall@k and explain its limits.


## Retrieval Failure Modes

A RAG system can fail before the model ever writes an answer. The retriever may miss the right document, return a weakly related document, or rank the correct source below irrelevant context. The generator can also fail after retrieval by ignoring evidence, dropping citations, or answering beyond the context.

For this reason, retrieval evaluation and generated-answer evaluation should stay separate. Recall@k tells us whether the expected document appears in the top results. Citation and abstention checks tell us whether the generation path respects the evidence that was retrieved.

The diagram below maps these failure points onto a full RAG pipeline. Some happen while documents are indexed, most at query time. This notebook's pipeline is simpler (no query rewriter or reranker), but the same failure points apply.

<div align="center">
  <img src="assets/03-rag-failure-points.png" alt="Failure points in the indexing and query processes of a RAG system" width="900" style="max-width: 100%; height: auto;" />
  <p><em>Failure points (red) in the indexing and query processes of a RAG system. Source: <a href="https://arxiv.org/abs/2401.05856" target="_blank" rel="noopener noreferrer">Barnett et al., 2024</a>, CC BY 4.0</em></p>
</div>


## Retrieval Terms In Plain Language

A **query** is the text used to search the corpus. It can be the user question directly, or a rewritten version of it. A **chunk** is a smaller piece of a document that can be retrieved independently. A **score** estimates how similar or relevant a document is to the query. A **top-k result** is one of the first `k` records returned by the retriever.

The notebook uses simple token counts instead of embeddings so the mechanics stay visible. In production, the same concepts usually appear with embedding models and vector databases: query embedding, chunk embedding, similarity score, and top-k retrieval.


## Retrieval Settings Change Accuracy

RAG quality depends on whether the right evidence reaches the model. A fluent generated answer is still weak if retrieval missed the supporting passage.

The chart below comes from a Hugging Face cookbook experiment that improved one RAG setting at a time. Tuning the chunk size gave the largest jump, and better embeddings and reranking added more, all before the reader LLM was changed. Retrieval settings are measurable choices, not decoration around the LLM call.

<div align="center">
  <img src="assets/03-rag-settings-accuracy.png" alt="Accuracy of different RAG configurations" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Accuracy of a RAG system as retrieval and reader settings improve one at a time. Source: <a href="https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/RAG_settings_accuracy.png" target="_blank" rel="noopener noreferrer">Hugging Face Cookbook</a></em></p>
</div>


## Load Shared Data And Runtime Helpers


In [ ]:
import json
import re
from pprint import pprint
from typing import Any, Literal

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ReplySource,
    call_model,
    format_context,
    retrieve,
)

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | Model: {CONFIG.model} | Dry run: {CONFIG.dry_run}"
)

## Retrieve Policy Evidence

[`strategy_helpers.py`](strategy_helpers.py) implements the retriever once for the repository: tokenization, term-frequency vectors, cosine ranking, and source metadata for citations. It is deliberately transparent: sparse token cosine similarity lets us inspect failure modes before swapping in embeddings or a vector database. A short keyword query shows the ranking:

In [ ]:
retrieval_preview = retrieve("duplicate payment refund invoice", top_k=3)
[(row["doc_id"], round(row["score"], 3)) for row in retrieval_preview]

Now retrieve evidence for a real support request. The answer path below uses this question.

In [ ]:
question = REQUESTS[1]["question"]
retrieved = retrieve(question, top_k=3)
pprint([{**record, "score": round(record["score"], 3)} for record in retrieved])

## Call The Model With Citations

The prompt requires the answer to cite source IDs. If retrieval is weak, we abstain before calling the model; if the context does not answer the question, the model should abstain rather than invent a policy.

The answer schema makes grounding inspectable: the output must say what it answered from and whether it abstained.


In [ ]:
class CitedAnswer(BaseModel):
    """Structured representation of a grounded answer."""

    answer: str
    source_ids: list[str]
    status: Literal["answered", "abstained", "parse_failed", "invalid_citations"]
    response_source: ReplySource | Literal["not_called"] = "not_called"

Retrieve first and abstain before calling the model when the best evidence is too weak.


In [ ]:
def retrieve_or_abstain(
    question: str, *, threshold: float = 0.08
) -> list[dict[str, Any]]:
    """Return evidence only when the top retrieval score is high enough."""
    evidence = retrieve(question, top_k=3)

    if not evidence or evidence[0]["score"] < threshold:
        return []

    return evidence

Build the cited-answer prompt separately from the RAG control flow.


In [ ]:
def rag_messages(question: str, evidence: list[dict[str, Any]]) -> list[dict[str, str]]:
    """Create a context-grounded prompt for cited JSON output."""
    context = format_context(evidence)
    return [
        {
            "role": "system",
            "content": (
                "Answer only from the context. Return JSON with answer, source_ids, and status. "
                "End each sentence of the answer with the ID of the context it comes from in square brackets, "
                'for example: "Late fees apply after 14 days [billing-terms]." List the same IDs in source_ids. '
                "Use status answered, or abstained with no citations if evidence is insufficient."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

The dry response mirrors the expected cited JSON structure.


In [ ]:
def dry_cited_answer(evidence: list[dict[str, Any]]) -> str:
    """Create deterministic cited output for dry-run execution."""
    return json.dumps(
        {
            "answer": f"{evidence[0]['text']} [{evidence[0]['doc_id']}]",
            "source_ids": [evidence[0]["doc_id"]],
            "status": "answered",
        }
    )

`rag_answer` puts the pieces together: it abstains on weak evidence, calls the provider, and validates both the JSON response and its citation IDs against the retrieved context. Valid IDs do not prove that the cited passages support every claim.


In [ ]:
def rag_answer(question: str, *, threshold: float = 0.08) -> CitedAnswer:
    """Retrieve policy context and ask the provider for a cited answer."""
    evidence = retrieve_or_abstain(question, threshold=threshold)
    if not evidence:
        return CitedAnswer(
            answer="I do not know from the available policy documents.",
            source_ids=[],
            status="abstained",
        )

    reply = call_model(
        rag_messages(question, evidence),
        dry_response=dry_cited_answer(evidence),
        response_format={"type": "json_object"},
    )

    try:
        result = CitedAnswer.model_validate_json(reply.text)
        result.response_source = reply.source
    except ValidationError:
        return CitedAnswer(
            answer=reply.text,
            source_ids=[],
            status="parse_failed",
            response_source=reply.source,
        )

    cited_ids = set(re.findall(r"\[([^\[\]]*)\]", result.answer))
    declared_ids = set(result.source_ids)
    context_ids = {record["doc_id"] for record in evidence}
    valid_citations = (
        bool(cited_ids) and cited_ids == declared_ids and cited_ids <= context_ids
        if result.status == "answered"
        else result.status == "abstained" and not (cited_ids or declared_ids)
    )
    if not valid_citations:
        return CitedAnswer(
            answer="The generated answer failed citation validation. Review the evidence.",
            source_ids=[],
            status="invalid_citations",
            response_source=reply.source,
        )
    return result

Run the grounded path once so the cited response can be inspected before evaluating retrieval quality.


In [ ]:
answer = rag_answer(question)
pprint(answer.model_dump())

Now ask a question the policy documents do not cover. Its best retrieval score is below the default threshold of 0.08, so `rag_answer` abstains without calling the model. For comparison, every real support request scores at least 0.23.

In [ ]:
unsupported_question = "Where should I park my bicycle near campus?"
best_match = retrieve(unsupported_question, top_k=1)[0]
print(f"Best match: {best_match['doc_id']} (score {best_match['score']:.3f})\n")
pprint(rag_answer(unsupported_question).model_dump())

## Evaluate Retrieval Recall

Before judging generation, we check whether retrieval found the expected source. If the right document is absent, the answer is already constrained by poor evidence.

**Recall@k** asks a narrow question: did the expected source appear somewhere in the top `k` retrieved records? If `recall@1` is low but `recall@3` is high, the retriever often finds the right document but ranks it below other documents. That may still be usable if the generator receives all three records, but it increases the risk of distraction.

Recall does not judge whether the final answer is well written, complete, or safe, and it does not prove that the cited sentence supports every claim. Those need checks on the generated answer, such as the citation validation in `rag_answer`.

In [ ]:
def recall_at_k(k: int = 3) -> float:
    """Measure whether expected policy documents appear in top-k retrieval."""
    hits = 0
    for request in REQUESTS:
        retrieved_ids = [
            record["doc_id"] for record in retrieve(request["question"], top_k=k)
        ]
        hits += request["expected_doc"] in retrieved_ids
    return hits / len(REQUESTS)


for k in [1, 2, 3]:
    print(f"recall@{k} = {recall_at_k(k):.2f}")

### Exercise: Find The Ranking Miss

`recall@1` is lower than `recall@2`, so at least one request does not get its expected document ranked first. For each request, compare the top-ranked document with `expected_doc` and print the requests where they differ, together with the document that was ranked first instead.


In [ ]:
# @TODO: Find the requests whose expected document is not ranked first.


<details>
<summary>Show Reference Solution</summary>

```python
for request in REQUESTS:
    top_result = retrieve(request["question"], top_k=1)[0]
    if top_result["doc_id"] != request["expected_doc"]:
        print(
            f"{request['request_id']}: expected {request['expected_doc']}, "
            f"ranked first {top_result['doc_id']}"
        )
```

</details>


Only `q6` misses: the prompt-injection question ranks the extension policy first. That document shares mostly common words with the question, such as "a", "the", and "should", plus "policy", while the correct document shares the meaningful ones: "ignore", "prompt", "retrieved", and "system". Raw term counts weight every word equally, so common words can decide the ranking. Weighting schemes such as TF-IDF or BM25, or embeddings, reduce this problem.


## Check Your Understanding

1. If the retriever misses the correct document, should the generator still answer? How does `rag_answer` handle weak evidence?
2. Should retrieval quality be checked before or after judging the generated answer?
3. An answer cites only source IDs that appear in the retrieved context. Does that prove the answer is correct?
4. If `recall@2` is higher than `recall@1`, what changed?

<details>
<summary>Show Reference Answers</summary>

1. No. Without the right evidence the answer would not be grounded, so the system should abstain or ask for more context. `rag_answer` checks the top retrieval score first: if it is below the threshold, it returns an abstention without calling the model.
2. Before. If the right evidence is missing, a fluent generated answer is not trustworthy.
3. No. Citation validation only shows that the IDs come from the retrieved context. The cited passages may still not support every claim, so answers also need grounding checks.
4. The correct document was not always ranked first, but it appeared when the retriever was allowed to return two documents.

</details>


## Further Reading

- [Seven Failure Points When Engineering a Retrieval Augmented Generation System](https://arxiv.org/abs/2401.05856) - Barnett et al., 2024
- [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172) - Liu et al., 2023
